---

# Seleciona flashes por Estado e Município

---

- `OBJETIVO`:
> Este código seleciona os flashes do GLM do satélite GOES-19 de um determinado dia por Estado. Ele também seleciona os flashes de todos os municípios de um mesmo Estado.


- `DADOS DE ENTRADA`:
> Arquivo CSV de flashes acumulados para um determinado dia. Exemplo de nome do arquivo: `flash_glm_goes_2026-09-21.csv`


- `DADOS DE SAÍDA`:
> 1. Tabela com a quantidade de flashes por Estado e por município de um mesmo estado. Exemplo: `flash_counts_by_state_2026-09-21.csv` e `flash_counts_by_municipio_RS_2026-09-21.csv`


- `OBSERVAÇÕES`:
   > Sem observações

- `REALIZADO POR`:
> Enrique V. Mattos - 23/09/2026

- `ATUALIZADO POR`:
> Enrique V. Mattos - 23/09/2026
---


# **1° Passo:** Preparando Ambiente

In [1]:
# instalações
!pip install -q ultraplot cartopy geobr salem rasterio pyproj geopandas descartes

# monta o drive
from google.colab import drive
drive.mount('/content/drive')

# diretório raiz
dir = '/content/drive/MyDrive/2-PESQUISA/0_GLM/estudos_de_caso/2026-09-20e21_FRENTEFRIA_RS'

# diretório de entrada
dir_input = f'{dir}/output/glm_diario_goes'

# diretório de saída
dir_output = f'{dir}/output/glm_figuras_goes'

# importa bibliotecas
import time
import geopandas as gpd
import geobr
import ultraplot as uplt
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
import pandas as pd
from datetime import timedelta, datetime
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.9/13.9 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.9 MB/s eta 0:00:00
Mounted at /content/drive


# Processa por `Estado`

In [2]:
%%time
#========================================================================================================================#
#                                             LIMITES DO BRASIL
#========================================================================================================================#
# limites do Brasil
lonmin_BR, lonmax_BR, latmin_BR, latmax_BR = -75.0, -33.0, -35.0, 7.0

#========================================================================================================================#
#                                              CARREGA SHAPEFILES
#========================================================================================================================#
# carrega o shapefile dos estados brasileiros. Usa a função read_state do geobr para obter os polígonos dos estados
shapefile_estados = geobr.read_state(year=2020)

#========================================================================================================================#
#                                   LEITURA DO ARQUIVO CSV DIÁRIO DE FLASHES
#========================================================================================================================#
# data
ano, mes, dia = '2026', '09', '21'

# leitura da planilha CSV
if dia == '20': df_flash = pd.read_csv(f'{dir_input}/flash_glm_goes_2026-09-20.csv')
if dia == '21': df_flash = pd.read_csv(f'{dir_input}/flash_glm_goes_2026-09-21.csv')
if dia == '22': df_flash = pd.read_csv(f'{dir_input}/flash_glm_goes_2026-09-22.csv')

# transforma a coluna data para índice do dataframe
df_flash.set_index('time', inplace=True)

# ordena o dataframe
df_flash.sort_index(inplace=True)

#========================================================================================================================#
#                                        SELECIONA OS FLASHES DO BRASIL
#========================================================================================================================#
# seleciona os flashes da região de interesee
df_flash_filtered = df_flash[ (df_flash['lon'] > lonmin_BR) & (df_flash['lon'] < lonmax_BR) & (df_flash['lat'] > latmin_BR) & (df_flash['lat'] < latmax_BR)]

# transforma o dataframe para array
lons_flash_filtered, lats_flash_filtered = df_flash_filtered['lon'].values, df_flash_filtered['lat'].values

#========================================================================================================================#
#                                 EXTRAI OS FLASHES QUE ESTÃO DENTRO DOS ESTADOS
#========================================================================================================================#
# novo geodataframe
df_flash_filtered_gpd = gpd.GeoDataFrame(df_flash_filtered.reset_index(),
                                         geometry = gpd.points_from_xy(df_flash_filtered.lon,
                                                                       df_flash_filtered.lat))

# Dicionário para armazenar o total de flashes por estado
flash_counts_by_state = {}

# Itera sobre cada estado no shapefile_estados
for index, state in shapefile_estados.iterrows():

    print('PROCESSANDO ===>>> ESTADO == ', state['name_state'])

    # Aplica a máscara para o estado atual, garantindo que 'state' seja um GeoDataFrame com CRS
    df_flash_in_state = gpd.overlay(df_flash_filtered_gpd, shapefile_estados.loc[[index]], how='intersection')

    # Conta o número de flashes neste estado
    num_flashes = len(df_flash_in_state)

    # Armazena a contagem no dicionário
    flash_counts_by_state[state['abbrev_state']] = num_flashes
    print('..... TOTAL DE FLASHES = ', num_flashes, '\n')

# colocar num dataframe e imprimir arquivo CSV
df_flash_counts_by_state = pd.DataFrame(list(flash_counts_by_state.items()), columns=['Estado', 'Total de Flashes'])

# organizar do maior para o menor da coluna "Total de Flashes"
df_flash_counts_by_state = df_flash_counts_by_state.sort_values(by='Total de Flashes', ascending=False)

# salvar arquivo CSV
df_flash_counts_by_state.to_csv(f'{dir_output}/flash_counts_by_state_{ano}-{mes}-{dia}.csv', index=False)

# mostra os dados
df_flash_counts_by_state

states_2020_simplified.parquet: 100%|██████████| 1.72M/1.72M [00:00<00:00, 28.7MB/s]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

PROCESSANDO ===>>> ESTADO ==  Rondônia
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Acre
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Amazonas
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Roraima
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Pará
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Amapá
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Tocantins
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Maranhão
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Piauí
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Ceará
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Rio Grande do Norte
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Paraíba
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Pernambuco
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Alagoas
..... TOTAL DE FLASHES =  0 

PROCESSANDO ===>>> ESTADO ==  Sergipe
..

,Estado,Total de Flashes
22,RS,414279
21,SC,134746
20,PR,119248
23,MS,25815
24,MT,23926
19,SP,9695
16,MG,2221
25,GO,1586
0,RO,0
1,AC,0


# Processa por `Município`

In [3]:
%%time
#========================================================================================================================#
#                                             LIMITES DO BRASIL
#========================================================================================================================#
lonmin_BR, lonmax_BR, latmin_BR, latmax_BR = -75.0, -33.0, -35.0, 7.0

#========================================================================================================================#
#                                      ESCOLHA DO ESTADO PELO USUÁRIO
#========================================================================================================================#
# digite a sigla do estado (ex: 'RS', 'SP', 'MS', 'MT', ...)
estado_escolhido = 'RS'   # <-- ALTERE AQUI

# carrega shapefile de todos os estados do Brasil
shapefile_estados = geobr.read_state(year=2020)

# filtra o estado escolhido
shapefile_estado = shapefile_estados[shapefile_estados['abbrev_state'] == estado_escolhido]

if shapefile_estado.empty:
    raise ValueError(f"Estado '{estado_escolhido}' não encontrado. Use a sigla (ex: RS, SP, MS).")

nome_estado = shapefile_estado['name_state'].values[0]
print(f'Estado escolhido: {nome_estado} ({estado_escolhido})')

#========================================================================================================================#
#                               CARREGA MUNICÍPIOS APENAS DO ESTADO ESCOLHIDO
#========================================================================================================================#
# carrega shapefile de municípios do estado escolhido
shapefile_municipios = geobr.read_municipality(year=2020, code_muni=None)

# filtra apenas os municípios do estado escolhido
shapefile_municipios_estado = shapefile_municipios[shapefile_municipios['abbrev_state'] == estado_escolhido].copy()

# total de municípios
tot_muni = len(shapefile_municipios_estado)

print(f'Total de municípios em {nome_estado}: {tot_muni}', '\n')

#========================================================================================================================#
#                                   LEITURA DO ARQUIVO CSV DIÁRIO DE FLASHES
#========================================================================================================================#
ano, mes, dia = '2026', '09', '21'

if dia == '20': df_flash = pd.read_csv(f'{dir_input}/flash_glm_goes_2026-09-20.csv')
if dia == '21': df_flash = pd.read_csv(f'{dir_input}/flash_glm_goes_2026-09-21.csv')
if dia == '22': df_flash = pd.read_csv(f'{dir_input}/flash_glm_goes_2026-09-22.csv')

df_flash.set_index('time', inplace=True)
df_flash.sort_index(inplace=True)

#========================================================================================================================#
#                                        SELECIONA OS FLASHES DO BRASIL
#========================================================================================================================#
df_flash_filtered = df_flash[ (df_flash['lon'] > lonmin_BR) & (df_flash['lon'] < lonmax_BR) &
                              (df_flash['lat'] > latmin_BR) & (df_flash['lat'] < latmax_BR)]

#========================================================================================================================#
#                                 EXTRAI OS FLASHES QUE ESTÃO DENTRO DOS MUNICÍPIOS
#========================================================================================================================#
# converte para GeoDataFrame
df_flash_filtered_gpd = gpd.GeoDataFrame(df_flash_filtered.reset_index(), geometry=gpd.points_from_xy(df_flash_filtered.lon, df_flash_filtered.lat), crs='EPSG:4326')

# garante mesmo CRS
shapefile_municipios_estado = shapefile_municipios_estado.to_crs(df_flash_filtered_gpd.crs)

# Dicionário para armazenar o total de flashes por município
flash_counts_by_municipio = {}

# Itera sobre cada município do estado escolhido
i=0
for index, muni in shapefile_municipios_estado.iterrows():

    nome_muni = muni['name_muni']
    print(f'[{i+1} de {tot_muni}] ... MUNICÍPIO = {nome_muni}', )

    # interseção dos flashes com o município atual
    df_flash_in_muni = gpd.overlay(df_flash_filtered_gpd, shapefile_municipios_estado.loc[[index]], how='intersection')

    num_flashes = len(df_flash_in_muni)
    flash_counts_by_municipio[nome_muni] = num_flashes
    print(f'{' ' * 15}TOTAL DE FLASHES = {num_flashes}\n')

    i+=1

#========================================================================================================================#
#                                       COLOCA NUM DATAFRAME E SALVA ARQUIVO CSV
#========================================================================================================================#
# monta e ordena a tabela
df_flash_counts_by_municipio = pd.DataFrame(list(flash_counts_by_municipio.items()), columns=['Município', 'Total de Flashes'])
df_flash_counts_by_municipio = df_flash_counts_by_municipio.sort_values(by='Total de Flashes', ascending=False)

# salva CSV
df_flash_counts_by_municipio.to_csv(f'{dir_output}/flash_counts_by_municipio_{estado_escolhido}_{ano}-{mes}-{dia}.csv', index=False)

print(f'\nTotal de relâmpagos por município em {nome_estado}:')
df_flash_counts_by_municipio

Estado escolhido: Rio Grande do Sul (RS)


municipalities_2020_simplified.parquet: 100%|██████████| 20.2M/20.2M [00:00<00:00, 70.1MB/s]


Total de municípios em Rio Grande do Sul: 497 

[1 de 497] ... MUNICÍPIO = Aceguá
               TOTAL DE FLASHES = 1799

[2 de 497] ... MUNICÍPIO = Água Santa
               TOTAL DE FLASHES = 158

[3 de 497] ... MUNICÍPIO = Agudo
               TOTAL DE FLASHES = 1431

[4 de 497] ... MUNICÍPIO = Ajuricaba
               TOTAL DE FLASHES = 204

[5 de 497] ... MUNICÍPIO = Alecrim
               TOTAL DE FLASHES = 3

[6 de 497] ... MUNICÍPIO = Alegrete
               TOTAL DE FLASHES = 7459

[7 de 497] ... MUNICÍPIO = Alegria
               TOTAL DE FLASHES = 34

[8 de 497] ... MUNICÍPIO = Almirante Tamandaré do Sul
               TOTAL DE FLASHES = 373

[9 de 497] ... MUNICÍPIO = Alpestre
               TOTAL DE FLASHES = 354

[10 de 497] ... MUNICÍPIO = Alto Alegre
               TOTAL DE FLASHES = 330

[11 de 497] ... MUNICÍPIO = Alto Feliz
               TOTAL DE FLASHES = 189

[12 de 497] ... MUNICÍPIO = Alvorada
               TOTAL DE FLASHES = 111

[13 de 497] ... MUNICÍPIO = Am

,Município,Total de Flashes
368,Sant'ana do Livramento,13521
385,São Gabriel,12926
383,São Francisco de Assis,12595
372,Santiago,8822
59,Cacequi,7976
...,...,...
326,Porto Lucena,4
4,Alecrim,3
463,Tucunduva,2
327,Porto Mauá,0
